# LLaMA-2 7B Instruction Fine-Tuning with QLoRA

A reproducible walkthrough of dataset preparation, 4-bit model loading, LoRA configuration, supervised fine-tuning, adapter export, and qualitative inference. **Requires an authorized Hugging Face account, a CUDA GPU, and the training CSV.**

> This notebook documents an experimental project, not a production-ready model. Training and evaluation have not been rerun as part of this GitHub cleanup.


## 1. Environment setup

Run in a GPU-enabled Colab environment. Package APIs change; if a newer release breaks compatibility, pin and document working versions.


In [ ]:
!pip -q install -U "transformers>=4.41.0,<6" "sentence-transformers>=5.2.0"
!pip -q install -U trl peft accelerate datasets pyarrow bitsandbytes

In [ ]:
import transformers, trl
print(transformers.__version__)
print(trl.__version__)


In [ ]:
from trl import SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig


In [ ]:
import os
import torch
from datasets import Dataset
import pandas as pd

## 2. Hugging Face access

The Meta LLaMA-2 base model is gated. Request access from the model publisher and authenticate using the interactive login flow. **Never commit tokens or credentials.**


In [ ]:
from huggingface_hub import login
login()


## 3. Prepare the instruction dataset

Upload `trainpt2.csv` into `/content/` or change `csv_path` to your dataset location. The CSV needs either a `text` column containing formatted examples or `question` and `answer` columns. The dataset is not distributed in this repository.


In [ ]:
csv_path = "/content/trainpt2.csv"  # upload your file in Colab and confirm path

df = pd.read_csv(csv_path)

if "text" not in df.columns:
    # Build a text column from question/answer
    required = {"question", "answer"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV must contain either 'text' OR both {required}. Missing: {missing}")

    df["text"] = (
        "###Human:\n" + df["question"].astype(str).str.strip() +
        "\n\n###Assistant:\n" + df["answer"].astype(str).str.strip()
    )

df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].astype(str).str.replace("\r\n", "\n")

dataset = Dataset.from_pandas(df[["text"]])
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = dataset["train"], dataset["test"]

print("Train size:", len(train_ds), "Eval size:", len(eval_ds))
print(train_ds[0]["text"][:400])

## 4. Load the quantized base model and configure LoRA

The base model is loaded with 4-bit NF4 quantization. LoRA adapters are trained instead of updating all base-model parameters.


In [ ]:
model_name = "meta-llama/Llama-2-7b-chat-hf"  # gated: you must be logged into HF in Colab
out_dir = "llama2-7b-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Explicitly set padding side for causal LMs

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
)
model.config.use_cache = False  # important for training

# Ensure model's pad_token_id is set correctly
if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
else:
    # Fallback if tokenizer.pad_token_id somehow remains None (shouldn't happen with the above if block)
    model.config.pad_token_id = model.config.eos_token_id

In [ ]:
peft_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)


## 5. Completion-only loss masking

The custom collator masks prompt tokens, so the training loss is applied to the assistant response. Samples missing the assistant marker are fully masked; check the formatting before training.


In [ ]:
import torch

class CompletionOnlyDataCollator:
    """
    Masks labels so loss is only computed on tokens after `response_template`.
    Compatible with SFTTrainer (creates attention_mask if missing).
    """

    def __init__(self, tokenizer, response_template="###Assistant:\n"):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def __call__(self, features):
        # SFTTrainer may NOT provide attention_mask → create it if missing
        for f in features:
            if "attention_mask" not in f:
                f["attention_mask"] = [1] * len(f["input_ids"])

        max_length_in_batch = max(len(f["input_ids"]) for f in features)

        input_ids_padded = []
        attention_mask_padded = []
        labels_masked = []

        for f in features:
            input_ids = f["input_ids"]
            attention_mask = f["attention_mask"]

            current_len = len(input_ids)
            padding_len = max_length_in_batch - current_len

            # ---- PAD ----
            padded_ids = list(input_ids) + [self.tokenizer.pad_token_id] * padding_len
            padded_mask = list(attention_mask) + [0] * padding_len

            input_ids_padded.append(padded_ids)
            attention_mask_padded.append(padded_mask)

            # ---- LABELS ----
            labels = list(input_ids).copy()

            start_idx = self._find_sublist(labels, self.response_template_ids)

            if start_idx == -1:
                # No assistant marker → don't train on this sample
                labels = [-100] * current_len
            else:
                end_of_marker = start_idx + len(self.response_template_ids)
                labels[:end_of_marker] = [-100] * end_of_marker

            labels_masked.append(labels + [-100] * padding_len)

        return {
            "input_ids": torch.tensor(input_ids_padded, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_padded, dtype=torch.long),
            "labels": torch.tensor(labels_masked, dtype=torch.long),
        }

    @staticmethod
    def _find_sublist(lst, sub):
        if not sub or len(sub) > len(lst):
            return -1
        for i in range(len(lst) - len(sub) + 1):
            if lst[i:i+len(sub)] == sub:
                return i
        return -1

collator = CompletionOnlyDataCollator(
    tokenizer=tokenizer,
    response_template="###Assistant:\n"
)

In [ ]:
sample = train_ds[0]["text"]
assert "###Assistant:" in sample, "Your training text must contain the response marker '###Assistant:'"


## 6. Supervised fine-tuning

This configuration uses one epoch, gradient accumulation, and a 5% evaluation split. Note: `eval_steps=200` can exceed the number of training steps on a small dataset; do not assume validation metrics were recorded unless they appear in actual run logs.


In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=out_dir,
    max_length=1024,          # replaces max_seq_length
    dataset_text_field="text",
    packing=False,

    # training args (move what you had in TrainingArguments here)
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    fp16=False,
    bf16=True,
    report_to="none",
    optim="adamw_bnb_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
    data_collator=collator        # your CompletionOnlyDataCollator stays
)

In [ ]:
trainer.train()

## 7. Save and reload the adapter

Save the LoRA adapter and tokenizer locally, then reload them for inference. This does **not** save the full 7B base model.


In [ ]:
trainer.model.save_pretrained(out_dir)   # LoRA adapter
tokenizer.save_pretrained(out_dir)
print("✅ Saved adapter to:", out_dir)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_name = model_name          # "meta-llama/Llama-2-7b-chat-hf"
adapter_dir = out_dir

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()


## 8. Qualitative inference examples

These prompts demonstrate generation, not quantitative model accuracy. Assess outputs against held-out reference answers before making performance claims.


In [ ]:
# Use the adapter-loaded model and tokenizer from the previous cell
agent = model
agent.eval()
print("Adapter model ready on:", agent.device)


In [ ]:
prompt = (
    "###Human:\n"
    "Why are traditional blockchain-based solutions not always practical for IoT applications?.\n\n"
    "###Assistant:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(agent.device)

with torch.inference_mode():
    output = agent.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
prompt = (
    "###Human:\n"
    "	How do signature aggregations work within the framework?.\n\n"
    "###Assistant:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(agent.device)

with torch.inference_mode():
    output = agent.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

## 9. Optional: publish the adapter

Only run this cell if you want to publish the adapter to the specified Hugging Face repository and have permission to do so. Review repository visibility and licensing first.


In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path=out_dir,
    repo_id="Alexis987/llama2-7b-lora",
    repo_type="model",
)


## Limitations and next steps

- The dataset is not included; provide your own authorized training CSV.
- No verified accuracy benchmark, held-out metric table, or reproducible run log is included here.
- Test on unseen domain questions and compare answers against references.
- Record exact package versions, GPU type, runtime, and seed for reproducibility.
- Confirm Meta model license terms before distributing model artifacts.
